# NB_LIBRO_04 · Conteo clásico de aves, con y sin corrección del fondo

**Libro:** *AgroVisión: del píxel a la decisión* · capítulo 4, sección 4.6

Repite el conteo clásico del notebook anterior y añade una variante más exigente: antes del umbral de Otsu se resta el fondo local con una transformación de sombrero de copa (*top-hat*), para que la iluminación desigual y la cama clara no entren en la máscara.

**Usa (solo lectura):** `AgroUTMACH_PIX/01_DATASET_PIO/PIO_TRAIN_VAL/data` (imágenes y etiquetas de `train` y `val`).

**Produce en `AGROVISION_LIBRO/`:** en `02_FIGURAS/cap04/` las figuras `Fig_4_5`, `Fig_4_6_correccion_fondo`, `Fig_4_7_conteo_por_semana_A`, `Fig_4_7_conteo_por_semana_C` y `Fig_4_8_conteo_casos`; en `03_VALORES/` el archivo `valores_cap04_conteo.json` y dos tablas CSV.

**Cómo ejecutarlo:** *Entorno de ejecución → Ejecutar todo*. No necesita GPU. Tarda entre 10 y 15 minutos. Al terminar, ejecute `NB_LIBRO_00_Empaquetar_resultados_v2` y suba el zip al chat.

In [ ]:
import os, re, json, math
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
    EN_COLAB = True
except ImportError:
    EN_COLAB = False

BASE = Path(os.environ.get('AGROVISION_BASE', '/content/drive/MyDrive'))
DIR_PIO = BASE / 'AgroUTMACH_PIX' / '01_DATASET_PIO' / 'PIO_TRAIN_VAL' / 'data'
DIR_LIBRO = BASE / 'AGROVISION_LIBRO'
DIR_FIG4 = DIR_LIBRO / '02_FIGURAS' / 'cap04'
DIR_VAL = DIR_LIBRO / '03_VALORES'
for d in (DIR_FIG4, DIR_VAL):
    d.mkdir(parents=True, exist_ok=True)

# --- Parámetros que puede cambiar -------------------------------------------
IMG_POLLOS = None     # con None, la imagen de validación con el número mediano de aves anotadas
N_CALIBRA = 20        # imágenes de entrenamiento por semana para calibrar el área por ave
SEMILLA = 2026
# -----------------------------------------------------------------------------
assert (DIR_PIO / 'images' / 'val').exists(), f'No se encuentra el dataset PIO en: {DIR_PIO}'
print('Dataset PIO:', DIR_PIO, '| Salida:', DIR_LIBRO)

In [ ]:
import numpy as np
import pandas as pd
import cv2
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
from matplotlib.ticker import FuncFormatter
from PIL import Image, ImageOps

ANCHO_IN = 16.0 / 2.54
DPI = 300
TINTA = '#0b0b0b'; TINTA_2 = '#52514e'; GRIS_FONDO = '#c9c8c2'; ACENTO = '#2a78d6'; REJILLA = '#e4e3de'
plt.rcParams.update({
    'font.family': 'DejaVu Sans', 'font.size': 8, 'axes.titlesize': 8, 'axes.labelsize': 8,
    'xtick.labelsize': 7, 'ytick.labelsize': 7, 'legend.fontsize': 7,
    'axes.edgecolor': TINTA_2, 'axes.labelcolor': TINTA, 'text.color': TINTA,
    'xtick.color': TINTA_2, 'ytick.color': TINTA_2, 'axes.linewidth': 0.6,
    'axes.spines.top': False, 'axes.spines.right': False,
    'figure.dpi': 110, 'savefig.dpi': DPI, 'figure.facecolor': 'white', 'savefig.facecolor': 'white',
})
EXT_IMG = {'.jpg', '.jpeg', '.png', '.bmp'}
VALORES = {}
FIGURAS = []


def guardar(fig, ruta):
    fig.savefig(ruta, dpi=DPI, bbox_inches='tight', pad_inches=0.04)
    FIGURAS.append(str(ruta)); print('Guardada:', ruta.name)


def leer_rgb(ruta):
    with Image.open(ruta) as im:
        return np.asarray(ImageOps.exif_transpose(im).convert('RGB'))


def coma(x, dec=1):
    return f'{x:.{dec}f}'.replace('.', ',')


def sin_ejes(ax):
    ax.set_xticks([]); ax.set_yticks([])
    for borde in ax.spines.values():
        borde.set_visible(False)


def elemento(k):
    return cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k, k))

## 1. Inventario del dataset PIO

In [ ]:
PATRON_PIO = re.compile(r'^([CP])-W(\d+)', re.IGNORECASE)


def listar_pio(particion):
    filas_p = []
    for ruta in sorted((DIR_PIO / 'images' / particion).iterdir()):
        if ruta.suffix.lower() not in EXT_IMG:
            continue
        hallado = PATRON_PIO.match(ruta.name)
        filas_p.append({'ruta': str(ruta), 'archivo': ruta.name,
                        'galpon': hallado.group(1).upper() if hallado else '?',
                        'semana': int(hallado.group(2)) if hallado else 0,
                        'etiqueta': str(DIR_PIO / 'labels' / particion / (ruta.stem + '.txt'))})
    return pd.DataFrame(filas_p)


def contar_anotadas(ruta_txt):
    p = Path(ruta_txt)
    if not p.exists():
        return np.nan
    return sum(1 for linea in p.read_text().splitlines() if len(linea.split()) == 5)


pio_val = listar_pio('val')
pio_val['anotadas'] = [contar_anotadas(r) for r in pio_val['etiqueta']]
pio_val = pio_val.dropna(subset=['anotadas']).reset_index(drop=True)
pio_val['anotadas'] = pio_val['anotadas'].astype(int)
pio_train = listar_pio('train')
pio_train = pio_train[pio_train['semana'] > 0]
muestra_cal = pd.concat([g.sample(min(N_CALIBRA, len(g)), random_state=SEMILLA)
                         for _, g in pio_train.groupby('semana')]).reset_index(drop=True)
muestra_cal['anotadas'] = [contar_anotadas(r) for r in muestra_cal['etiqueta']]
muestra_cal = muestra_cal.dropna(subset=['anotadas']).reset_index(drop=True)
print(f'Validación: {len(pio_val)} imágenes · calibración: {len(muestra_cal)} imágenes de entrenamiento')

## 2. Segmentación y estimadores

`segmentar_aves(rgb, corregir=False)` es la cadena simple del capítulo. Con `corregir=True` se aplica antes del umbral la transformación de sombrero de copa: a la imagen en gris se le resta su apertura con un elemento grande, de modo que solo quedan los objetos claros más pequeños que ese elemento.

In [ ]:
ANCHO_PIO = 960
S_MAX = 90            # saturación máxima admitida (descarta comederos amarillos)
AREA_MIN = 20         # área mínima de un componente, en píxeles
K_AVES = 5            # apertura que elimina puntos sueltos
K_FONDO = 81          # elemento del sombrero de copa: mayor que un ave adulta en la imagen de 960 px


def leer_pio(ruta):
    im = leer_rgb(ruta)
    esc = ANCHO_PIO / im.shape[1]
    return cv2.resize(im, (ANCHO_PIO, round(im.shape[0] * esc)), interpolation=cv2.INTER_AREA)


def segmentar_aves(rgb, corregir=False):
    gris = cv2.GaussianBlur(cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY), (5, 5), 0)
    base = cv2.morphologyEx(gris, cv2.MORPH_TOPHAT, elemento(K_FONDO)) if corregir else gris
    t, binaria = cv2.threshold(base, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    sat = cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV)[:, :, 1]
    filtrada = binaria.copy(); filtrada[sat > S_MAX] = 0
    abierta = cv2.morphologyEx(filtrada, cv2.MORPH_OPEN, elemento(K_AVES))
    n, etiquetas, estad, _ = cv2.connectedComponentsWithStats(abierta, connectivity=8)
    areas = estad[1:, cv2.CC_STAT_AREA].astype(np.float64)
    validos = np.where(areas >= AREA_MIN)[0] + 1
    return {'umbral': float(t), 'gris': gris, 'base': base, 'binaria': binaria, 'abierta': abierta,
            'etiquetas': etiquetas, 'validos': validos, 'areas': areas[validos - 1]}


def estimar(sa):
    a = sa['areas']
    if len(a) == 0:
        return {'n_componentes': 0, 'n_area_ref': 0, 'area_primer_plano': 0.0, 'area_ref': np.nan, 'por_componente': a}
    ref = float(np.median(a))
    por_comp = np.maximum(1, np.round(a / ref))
    return {'n_componentes': int(len(a)), 'n_area_ref': int(por_comp.sum()), 'area_primer_plano': float(a.sum()),
            'area_ref': ref, 'por_componente': por_comp}


# Calibración del área por ave, por semana, para cada cadena
VARIANTES = {'s': False, 'c': True}          # s = cadena simple · c = con corrección del fondo
filas_calib = []
for k, fila in enumerate(muestra_cal.itertuples(index=False), start=1):
    im = leer_pio(fila.ruta)
    reg = {'semana': fila.semana, 'anotadas': fila.anotadas}
    for v, corr in VARIANTES.items():
        reg['area_' + v] = estimar(segmentar_aves(im, corr))['area_primer_plano']
    filas_calib.append(reg)
    if k % 30 == 0 or k == len(muestra_cal):
        print(f'  calibración: {k}/{len(muestra_cal)}')
calib = pd.DataFrame(filas_calib)
suma = calib.groupby('semana')[['anotadas', 'area_s', 'area_c']].sum()
area_por_ave = {v: (suma['area_' + v] / suma['anotadas']) for v in VARIANTES}
area_global = {v: float(calib['area_' + v].sum() / calib['anotadas'].sum()) for v in VARIANTES}
print(pd.DataFrame(area_por_ave).round(1).rename(columns={'s': 'px² por ave, simple', 'c': 'px² por ave, corregida'}).to_string())

# Evaluación en validación
filas_cnt = []
for k, fila in enumerate(pio_val.itertuples(index=False), start=1):
    im = leer_pio(fila.ruta)
    reg = {'archivo': fila.archivo, 'ruta': fila.ruta, 'galpon': fila.galpon, 'semana': fila.semana, 'anotadas': fila.anotadas}
    for v, corr in VARIANTES.items():
        sa = segmentar_aves(im, corr); e = estimar(sa)
        a_ref = float(area_por_ave[v].get(fila.semana, area_global[v]))
        reg.update({'A' + v: e['n_componentes'], 'B' + v: e['n_area_ref'],
                    'C' + v: int(round(e['area_primer_plano'] / a_ref)) if a_ref > 0 else 0,
                    'pp_' + v: e['area_primer_plano'] / sa['abierta'].size * 100, 'umbral_' + v: sa['umbral']})
    filas_cnt.append(reg)
    if k % 50 == 0 or k == len(pio_val):
        print(f'  validación: {k}/{len(pio_val)}')
cnt = pd.DataFrame(filas_cnt)
cnt.drop(columns=['ruta']).round(2).to_csv(DIR_VAL / 'conteo_clasico_v2_por_imagen.csv', index=False)

NOMBRES = {'As': 'A · Componentes (cadena simple)', 'Bs': 'B · Área de referencia (cadena simple)',
           'Cs': 'C · Área calibrada (cadena simple)', 'Ac': 'A · Componentes (con corrección)',
           'Bc': 'B · Área de referencia (con corrección)', 'Cc': 'C · Área calibrada (con corrección)'}


def errores(tabla, col):
    d = tabla[col] - tabla['anotadas']
    return {'EAM': round(float(d.abs().mean()), 1), 'sesgo': round(float(d.mean()), 1),
            'EPAM_pct': round(float((d.abs() / tabla['anotadas'].clip(lower=1)).mean() * 100), 1),
            'error_mediano': round(float(d.abs().median()), 1),
            'r': round(float(np.corrcoef(tabla[col], tabla['anotadas'])[0, 1]), 3) if tabla[col].std() > 0 else None}


err_global = {c: errores(cnt, c) for c in NOMBRES}
print('\nError global en validación (452 imágenes):')
print(pd.DataFrame(err_global).T.rename(index=NOMBRES).to_string())
err_semana = {int(s): {c: errores(g, c) for c in NOMBRES} | {'imagenes': int(len(g)), 'aves_media': round(float(g['anotadas'].mean()), 1)}
              for s, g in cnt.groupby('semana')}
err_galpon = {str(s): {c: errores(g, c) for c in NOMBRES} | {'imagenes': int(len(g)), 'aves_media': round(float(g['anotadas'].mean()), 1)}
              for s, g in cnt.groupby('galpon')}
tabla_sem = pd.DataFrame([{'semana': s, 'imagenes': v['imagenes'], 'aves_media': v['aves_media'],
                           **{c + ' EAM': v[c]['EAM'] for c in NOMBRES}, **{c + ' sesgo': v[c]['sesgo'] for c in NOMBRES}}
                          for s, v in err_semana.items()])
tabla_sem.to_csv(DIR_VAL / 'tabla_4_conteo_clasico_v2_por_semana.csv', index=False)
print('\nEAM por semana:'); print(tabla_sem[['semana', 'imagenes', 'aves_media'] + [c + ' EAM' for c in NOMBRES]].to_string(index=False))

n_real = cnt['anotadas']
media_sem = cnt.groupby('semana')['anotadas'].transform('mean')
MEJOR = min(('Ac', 'Bc', 'Cc'), key=lambda c: err_global[c]['EAM'])
print('\nMejor estimador con corrección:', NOMBRES[MEJOR])
VALORES['conteo_v2'] = {
    'parametros': {'ancho_px': ANCHO_PIO, 'saturacion_max': S_MAX, 'area_min_px': AREA_MIN, 'apertura': K_AVES,
                   'elemento_fondo_px': K_FONDO, 'imagenes_calibracion_por_semana': N_CALIBRA},
    'n_val': int(len(cnt)), 'aves_media_val': round(float(n_real.mean()), 1),
    'area_por_ave_px2': {v: {int(s): round(float(x), 1) for s, x in area_por_ave[v].items()} for v in VARIANTES},
    'error_global': err_global, 'error_por_semana': err_semana, 'error_por_galpon': err_galpon,
    'referencia_media_constante_EAM': round(float((n_real - n_real.mean()).abs().mean()), 1),
    'referencia_media_por_semana_EAM': round(float((n_real - media_sem).abs().mean()), 1),
    'pct_primer_plano_medio': {v: round(float(cnt['pp_' + v].mean()), 1) for v in VARIANTES},
    'pct_primer_plano_por_semana': {v: {int(s): round(float(x), 1) for s, x in cnt.groupby('semana')['pp_' + v].mean().items()} for v in VARIANTES},
    'correlacion_primer_plano_con_aves': {v: round(float(np.corrcoef(cnt['pp_' + v], n_real)[0, 1]), 3) for v in VARIANTES},
    'componentes_por_ave': {v: round(float(cnt['A' + v].sum() / n_real.sum()), 2) for v in VARIANTES},
    'mejor_con_correccion': MEJOR,
    'referencia_YOLO11s_NB08': {'EAM': 13.9, 'sesgo': 13.2, 'n_imagenes': 452},
}

## 3. Figuras 4.5 y 4.6 · La misma imagen con la cadena simple y con corrección del fondo

In [ ]:
if IMG_POLLOS:
    fila_pio = pio_val[pio_val['archivo'] == IMG_POLLOS].iloc[0]
else:
    fila_pio = pio_val.iloc[(pio_val['anotadas'] - pio_val['anotadas'].median()).abs().argsort().iloc[0]]
img_pio = leer_pio(fila_pio['ruta'])
prop_p = img_pio.shape[0] / img_pio.shape[1]
RAMPA = ['#a9cdf5', '#2a78d6', '#0d366b']           # 1 ave · 2 o 3 aves · 4 o más
print('Imagen de ejemplo:', fila_pio['archivo'], '· aves anotadas:', fila_pio['anotadas'])


def capa_componentes(rgb, sa, e):
    fondo_gris = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY).astype(np.float32) * 0.35 + 150
    capa = np.repeat(np.clip(fondo_gris, 0, 255)[:, :, None], 3, axis=2).astype(np.uint8)
    tabla_color = np.zeros((sa['etiquetas'].max() + 1, 3), np.uint8)
    usados = np.zeros(sa['etiquetas'].max() + 1, bool)
    for etq_c, n_c in zip(sa['validos'], e['por_componente']):
        tono = RAMPA[0] if n_c <= 1 else RAMPA[1] if n_c <= 3 else RAMPA[2]
        tabla_color[etq_c] = [int(tono[i:i + 2], 16) for i in (1, 3, 5)]
        usados[etq_c] = True
    dentro = usados[sa['etiquetas']]
    capa[dentro] = tabla_color[sa['etiquetas']][dentro]
    return capa


def leyenda_rampa(ax):
    ax.legend(handles=[Patch(facecolor=RAMPA[0], label='1 ave'), Patch(facecolor=RAMPA[1], label='2 o 3 aves'),
                       Patch(facecolor=RAMPA[2], label='4 o más')],
              loc='lower center', bbox_to_anchor=(0.5, -0.16), ncol=3, frameon=False, fontsize=6.5, handlelength=1.2)


info_ej = {}
for v, corr in VARIANTES.items():
    sa = segmentar_aves(img_pio, corr); e = estimar(sa)
    fig, axs = plt.subplots(2, 2, figsize=(ANCHO_IN, ANCHO_IN * prop_p + 0.75), constrained_layout=True)
    if not corr:
        axs[0, 0].imshow(img_pio); axs[0, 0].set_title(f"(a) Imagen original · {fila_pio['anotadas']} aves anotadas")
        axs[0, 1].imshow(sa['binaria'], cmap='gray'); axs[0, 1].set_title(f"(b) Umbral de Otsu, T = {int(sa['umbral'])}")
        axs[1, 0].imshow(sa['abierta'], cmap='gray'); axs[1, 0].set_title('(c) Sin píxeles saturados y tras la apertura')
        nombre_fig = 'Fig_4_5_conteo_clasico_pasos.png'
    else:
        axs[0, 0].imshow(sa['gris'], cmap='gray', vmin=0, vmax=255); axs[0, 0].set_title('(a) Imagen en escala de grises')
        axs[0, 1].imshow(sa['base'], cmap='gray', vmin=0, vmax=max(60, int(np.percentile(sa['base'], 99.8))))
        axs[0, 1].set_title(f'(b) Sombrero de copa, elemento de {K_FONDO} px')
        axs[1, 0].imshow(sa['abierta'], cmap='gray'); axs[1, 0].set_title(f"(c) Otsu (T = {int(sa['umbral'])}), saturación y apertura")
        nombre_fig = 'Fig_4_6_correccion_fondo.png'
    axs[1, 1].imshow(capa_componentes(img_pio, sa, e))
    axs[1, 1].set_title(f"(d) {e['n_componentes']} componentes · {e['n_area_ref']} aves estimadas por área")
    leyenda_rampa(axs[1, 1])
    for ax in axs.ravel():
        sin_ejes(ax)
    guardar(fig, DIR_FIG4 / nombre_fig)
    plt.show()
    fila_c = cnt[cnt['archivo'] == fila_pio['archivo']].iloc[0]
    info_ej[v] = {'umbral': int(sa['umbral']), 'componentes': int(e['n_componentes']), 'B_area_ref': int(e['n_area_ref']),
                  'C_area_calibrada': int(fila_c['C' + v]), 'area_referencia_px': round(float(e['area_ref']), 1),
                  'pct_primer_plano': round(float(fila_c['pp_' + v]), 1),
                  'componentes_1_ave': int((e['por_componente'] <= 1).sum()),
                  'componentes_2_3_aves': int(((e['por_componente'] > 1) & (e['por_componente'] <= 3)).sum()),
                  'componentes_4_o_mas': int((e['por_componente'] > 3).sum()),
                  'componente_mayor_aves': int(e['por_componente'].max()) if len(e['por_componente']) else 0}
VALORES['ejemplo'] = {'imagen': fila_pio['archivo'], 'semana': int(fila_pio['semana']), 'galpon': fila_pio['galpon'],
                      'aves_anotadas': int(fila_pio['anotadas']), 'simple': info_ej['s'], 'con_correccion': info_ej['c']}
print(json.dumps(VALORES['ejemplo'], ensure_ascii=False, indent=1))

## 4. Figura 4.7 · Estimación frente a anotación, por semana, sin y con corrección del fondo

In [ ]:
semanas = sorted(cnt['semana'].unique())
n_col = 3 if len(semanas) > 2 else len(semanas)
n_fil = math.ceil(len(semanas) / n_col)
for letra_est, nombre_est in (('A', 'número de componentes'), ('C', 'área calibrada por semana')):
    cs, cc = letra_est + 's', letra_est + 'c'
    fig, axs = plt.subplots(n_fil, n_col, figsize=(ANCHO_IN, ANCHO_IN / n_col * n_fil * 0.98 + 0.5),
                            constrained_layout=True, squeeze=False)
    lim_sup = float(max(cnt['anotadas'].max(), cnt[cs].quantile(0.99), cnt[cc].quantile(0.99))) * 1.05
    for ax, s in zip(axs.ravel(), semanas):
        g = cnt[cnt['semana'] == s]
        ax.plot([0, lim_sup], [0, lim_sup], color=TINTA_2, linewidth=0.7, linestyle='--', zorder=1)
        ax.scatter(g['anotadas'], g[cs], s=9, facecolors='none', edgecolors='#8a8984', linewidths=0.6, zorder=2)
        ax.scatter(g['anotadas'], g[cc], s=9, color=ACENTO, linewidths=0, zorder=3)
        ax.set_xlim(0, lim_sup); ax.set_ylim(0, lim_sup); ax.set_aspect('equal')
        ax.grid(color=REJILLA, linewidth=0.5); ax.set_axisbelow(True)
        ax.set_title((f'Semana {s}' if s else 'Semana sin identificar') + f' · n = {len(g)}', loc='left')
        eam_s = float((g[cs] - g['anotadas']).abs().mean()); eam_c = float((g[cc] - g['anotadas']).abs().mean())
        ax.text(0.04, 0.96, f'EAM simple = {coma(eam_s, 1)}\nEAM corregida = {coma(eam_c, 1)}', transform=ax.transAxes,
                va='top', ha='left', fontsize=6.3, linespacing=1.3, bbox=dict(facecolor='white', edgecolor='none', pad=1.5, alpha=0.9))
    for ax in axs.ravel()[len(semanas):]:
        ax.set_visible(False)
    for ax in axs[-1, :]:
        ax.set_xlabel('Aves anotadas')
    for ax in axs[:, 0]:
        ax.set_ylabel('Aves estimadas')
    fig.legend(handles=[Line2D([], [], marker='o', linestyle='', markersize=4, markerfacecolor='none', markeredgecolor='#8a8984',
                               label='Cadena simple'),
                        Line2D([], [], marker='o', linestyle='', markersize=4, color=ACENTO, label='Con corrección del fondo'),
                        Line2D([], [], color=TINTA_2, linewidth=0.7, linestyle='--', label='Estimación perfecta')],
               loc='outside lower center', ncol=3, frameon=False, fontsize=6.8)
    guardar(fig, DIR_FIG4 / f'Fig_4_7_conteo_por_semana_{letra_est}.png')
    plt.show()

## 5. Figura 4.8 · Cuatro casos del mejor estimador con corrección del fondo

In [ ]:
cnt['dif_mejor'] = cnt[MEJOR] - cnt['anotadas']
orden_err = cnt.assign(abs_m=cnt['dif_mejor'].abs()).sort_values('abs_m').reset_index(drop=True)
casos = [('Menor error', orden_err.iloc[0]), ('Error mediano', orden_err.iloc[len(orden_err) // 2]),
         ('Mayor subestimación', cnt.sort_values('dif_mejor').iloc[0]), ('Mayor sobreestimación', cnt.sort_values('dif_mejor').iloc[-1])]
fig, axs = plt.subplots(2, 2, figsize=(ANCHO_IN, ANCHO_IN * prop_p + 0.95), constrained_layout=True)
info_casos = {}
for ax, letra, (nombre, reg) in zip(axs.ravel(), 'abcd', casos):
    im = leer_pio(reg['ruta'])
    sa = segmentar_aves(im, True)
    ax.imshow(im); sin_ejes(ax)
    mascara_c = np.isin(sa['etiquetas'], sa['validos']).astype(np.uint8)
    contornos, _ = cv2.findContours(mascara_c, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    for c in contornos:
        pts = np.vstack([c[:, 0, :], c[:1, 0, :]])
        ax.plot(pts[:, 0], pts[:, 1], color=ACENTO, linewidth=0.55)
    sem_txt = f" · semana {reg['semana']}" if reg['semana'] else ''
    ax.set_title(f"({letra}) {nombre}{sem_txt}\n{reg['anotadas']} anotadas · {reg[MEJOR]} estimadas", fontsize=7.5, linespacing=1.2)
    info_casos[nombre] = {'archivo': reg['archivo'], 'semana': int(reg['semana']), 'galpon': reg['galpon'], 'anotadas': int(reg['anotadas']),
                          **{c: int(reg[c]) for c in NOMBRES}, 'pp_s': round(float(reg['pp_s']), 1), 'pp_c': round(float(reg['pp_c']), 1)}
guardar(fig, DIR_FIG4 / 'Fig_4_8_conteo_casos.png')
plt.show()
VALORES['casos'] = {'estimador': MEJOR, 'casos': info_casos}

VALORES['figuras_generadas'] = [Path(f).name for f in FIGURAS]
with open(DIR_VAL / 'valores_cap04_conteo.json', 'w', encoding='utf-8') as fh:
    json.dump(VALORES, fh, ensure_ascii=False, indent=2, default=lambda o: o.item() if hasattr(o, 'item') else str(o))
print(json.dumps(info_casos, ensure_ascii=False, indent=1))
print('\nFiguras generadas:', [Path(f).name for f in FIGURAS])
print('Siguiente paso: ejecute NB_LIBRO_00_Empaquetar_resultados_v2 y suba el zip al chat.')